# MerchAI — LightGBM Baseline

**Store:** CA_1  
**Horizon:** 28 days (matches M5 competition)  
**Metric:** WRMSSE (simplified equal-weight version)

This notebook runs the full pipeline: load → features → train → evaluate.

In [ ]:
import sys
sys.path.append('..')

import time
import numpy as np
import pandas as pd
import lightgbm as lgb

from src.data.loader import load_m5
from src.data.features import build_features
from src.forecast.evaluate import wrmsse

DATA_DIR = '../data/raw'
STORE    = 'CA_1'
HORIZON  = 28

## 1. Load Data

In [ ]:
t0 = time.time()
dfs = load_m5(DATA_DIR)
print(f'Loaded in {time.time()-t0:.1f}s')
for name, df in dfs.items():
    print(f'  {name:12s}  {df.shape}')

## 2. Build Features

In [ ]:
t0 = time.time()
df = build_features(dfs, store=STORE)
print(f'Features built in {time.time()-t0:.1f}s')
print(f'Rows: {len(df):,}  |  Columns: {df.shape[1]}')
df.head(3)

## 3. Train / Validation Split

Last 28 days = validation set. Matches the M5 competition horizon.

In [ ]:
max_day = df['day_num'].max()
cut     = max_day - HORIZON

train_df = df[df['day_num'] <= cut].copy()
val_df   = df[df['day_num'] >  cut].copy()

print(f'Train: days 1-{cut}  ({len(train_df):,} rows)')
print(f'Val:   days {cut+1}-{max_day}  ({len(val_df):,} rows)')

## 4. Train LightGBM

In [ ]:
FEATURE_COLS = [
    'item_id', 'dept_id', 'cat_id',
    'wday', 'month', 'year',
    'snap_CA', 'sell_price',
    'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_28',
]
TARGET = 'sales'

# Fill any residual NaNs in rolling means with the lag_28 value
for df_ in [train_df, val_df]:
    df_['rolling_mean_7']  = df_['rolling_mean_7'].fillna(df_['lag_28'])
    df_['rolling_mean_28'] = df_['rolling_mean_28'].fillna(df_['lag_28'])

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]
X_val   = val_df[FEATURE_COLS]
y_val   = val_df[TARGET]

params = {
    'objective':    'tweedie',
    'tweedie_variance_power': 1.1,
    'num_leaves':   127,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbose': -1,
}

dtrain = lgb.Dataset(X_train, label=y_train)
dval   = lgb.Dataset(X_val,   label=y_val, reference=dtrain)

t0 = time.time()
callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(50)]
model = lgb.train(
    params, dtrain,
    num_boost_round=500,
    valid_sets=[dval],
    callbacks=callbacks,
)
print(f'\nTrained in {time.time()-t0:.1f}s  |  Best round: {model.best_iteration}')

## 5. Feature Importance

In [ ]:
importance = pd.Series(
    model.feature_importance(importance_type='gain'),
    index=FEATURE_COLS
).sort_values(ascending=False)

print('Feature importance (gain):')
print(importance.to_string())

## 6. Evaluate on Validation Set

In [ ]:
preds = model.predict(X_val)
preds = np.clip(preds, 0, None)  # sales can't be negative

# WRMSSE (simplified equal-weight across all series in CA_1)
score = wrmsse(
    y_true=y_val.values,
    y_pred=preds,
    weights=np.ones(len(y_val)),
)

# Also compute plain RMSE for intuition
rmse = np.sqrt(np.mean((y_val.values - preds) ** 2))

print('=' * 40)
print(f'  WRMSSE (equal-weight):  {score:.4f}')
print(f'  RMSE:                   {rmse:.4f}')
print(f'  Mean actual sales/day:  {y_val.mean():.2f}')
print('=' * 40)
print()
print('M5 leaderboard reference: top 10% ~ WRMSSE 0.50')
print('Week 1 target: MAPE < 25% (any reasonable number is fine for now)')

## 7. Sample Predictions

A sanity check: do the numbers look like real sales?

In [ ]:
sample = val_df[['id', 'day_num', TARGET]].copy()
sample['predicted'] = preds.round(2)
sample['error']     = (sample[TARGET] - sample['predicted']).abs().round(2)

print('5 random validation rows:')
print(sample.sample(5, random_state=42).to_string(index=False))